In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# Shared Memory in CUDA


> **Overview**
>
> 
>  * **Time:** 30 min
> 
>  1. Learn about shared memory in CUDA.
>  1. Understand the characteristics and usage of shared memory.

Shared memory is a type of memory that is shared among threads within the same block in CUDA. It is much
faster than global memory and can be used to store data that needs to be accessed by multiple threads.
Shared memory is particularly useful for reducing global memory accesses and improving performance in
parallel algorithms.

## Shared Memory Characteristics

Shared memory in CUDA has several key characteristics:

1. **Scope**: Shared memory is accessible only to threads within the same block. Threads in different blocks cannot access each other's shared memory.

2. **Speed**: Shared memory is on-chip memory, making it much faster than global memory (DRAM). Accessing shared memory typically has a latency of around 10-50 cycles, compared to 400-800 cycles for global memory.

3. **Size**: The size of shared memory is limited, typically ranging from 48 KB to 96 KB per block, depending on the GPU architecture.

4. **Coherency**: Shared memory is coherent across all threads in a block, meaning that changes made by one thread are visible to others in the same block.

5. **Usage**: Shared memory is often used for data that needs to be accessed frequently by multiple threads, such as intermediate results in parallel computations.

**Shared Memory vs DRAM in NVIDIA GPUs**

| Feature | Shared Memory | DRAM (Global Memory) |
| --- | --- | --- |
| Access speed | Very fast (10–50 cycles) | Slow (400–800 cycles) |
| Capacity | Small (48–96 KB per SM) | Large (GBs) |
| Usage | Frequently accessed data within a block | Data shared across all blocks |
| Benefit | Reduces global memory traffic and accelerates access | Higher latency, but larger capacity |

There are two main types of shared memory in CUDA:

1. **Static Shared Memory**: This is defined at compile time and has a fixed size. It is declared using the `__shared__` keyword in the kernel code. Static shared memory is allocated for each block and is initialized when the block is launched.

```cpp
__global__ void addKernel(int *input, int *output) 
{
    __shared__ int sdata[256];  // static shared memory

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Load data from global to shared memory
    sdata[tid] = input[idx];
    __syncthreads();

    // Do something with shared memory
    sdata[tid] += 10;
    __syncthreads();

    // Store result back to global memory
    output[idx] = sdata[tid];
}
```

2. **Dynamic Shared Memory**: This is allocated at runtime and can vary in size based on the needs of the kernel. It is declared using the `extern __shared__` keyword in the kernel code. Dynamic shared memory allows for more flexibility, as its size can be specified when launching the kernel.

```cpp
__global__ void addKernel(int *input, int *output) 
{
    extern __shared__ int sdata[];  // dynamic shared memory
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Load data from global to shared memory
    sdata[tid] = input[idx];
    __syncthreads();

    // Do something with shared memory
    sdata[tid] += 10;
    __syncthreads();

    // Store result back to global memory
    output[idx] = sdata[tid];
}

int main() 
{
    int threads = 256;
    int sharedMemSize = threads * sizeof(int);  // Size of dynamic shared memory

    ...
    ...
    ...

    addKernel<<<1, threads, sharedMemSize>>>(d_input, d_output);
}
```

> **Important**
>
> You declare only one `extern __shared__ array` inside your kernel.

> **Explanation**
>
> 
>  `__syncthreads()` is a barrier synchronization function in CUDA that ensures all threads in the same 
>  thread block:
> 
>  1. Reach the barrier, and
>  2. Complete all memory accesses (reads/writes to shared memory)
> 
>  before any thread in the block continues past it.

> **Key Points**
>
> 
>  1. Shared memory is a fast, on-chip memory accessible by threads within the same block.
>  1. It is used to reduce global memory accesses and improve performance in parallel algorithms.
>  1. There are two types of shared memory: static (fixed size) and dynamic (runtime size).
>  1. The `__syncthreads()` function is used to synchronize threads within a block, ensuring all threads reach a certain point before proceeding.

## Working with the examples

The CUDA code above illustrates declarations and launch syntax. These are fragments: the dynamic-memory example uses `...` for omitted host setup and cannot be compiled as written. Both kernels assume their input/output buffers cover every launched thread; the static example also assumes at most 256 threads per block.

The first kernel uses a fixed-size shared array; the second receives the shared-memory size in bytes through the third launch argument. Threads must write the required values before reading them; shared arrays are not automatically zero-initialized.

## Exercise 07: Shared Memory

**Time: 30 min**

Adjust a dark grayscale image using its mean brightness. Threads cooperate through
a shared-memory reduction; the CPU combines block totals, then a second kernel
applies the gain. [Exercise instructions](../docs/source/tutorial/exercises/07-shared-memory.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

### Preview the input and CPU reference

These settings match the documented PBS job (`385 × 257`, target 128, 256 threads).
The partial final block tests zero contributions and participation in barriers.
Changing preview variables here does not change the job script. To change GPU
inputs/settings, edit `07-shared-memory.pbs` and keep the dimensions below aligned.

In [ ]:
import json
import math
import runpy

image_tools = runpy.run_path(str(repo_root / "exercises/07-shared-memory/image-tools.py"))
width, height = 385, 257
preview = image_tools["make_scene"](width, height, 0.35)
reference, mean, gain = image_tools["adjust"](preview, 128)
preview_dir = repo_root / "notebook-output/07-shared-memory"
preview_dir.mkdir(parents=True, exist_ok=True)
for name, pixels in [("input.png", preview), ("cpu-reference.png", reference)]:
    image_tools["write_png"](preview_dir / name, pixels, width, height)
    display(Image(filename=str(preview_dir / name)))
print(f"CPU input mean: {mean:.3f}; gain: {gain:.3f}")

### Inspect the reduction, then submit

<a href="../exercises/07-shared-memory/brightness.cu" target="_blank" rel="noopener noreferrer">Open brightness.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 07-shared-memory.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
show_job_output("07-shared-memory", job_id)

### Check and display the completed GPU results

This reads the actual input/output and statistics from the batch job, not a separate
notebook-generated input. It compares every block sum and adjusted pixel with the
CPU reference before displaying the result.

In [ ]:
job_output_dir = repo_root / "exercises/07-shared-memory"
paths = [job_output_dir / name for name in
         ["input.raw", "adjusted.raw", "stats.json", "block-sums.txt"]]
if outputs_ready(paths, submitted_at):
    input_pixels = (job_output_dir / "input.raw").read_bytes()
    gpu_pixels = (job_output_dir / "adjusted.raw").read_bytes()
    stats = json.loads((job_output_dir / "stats.json").read_text())
    block_sums = list(map(int, (job_output_dir / "block-sums.txt").read_text().split()))
    assert len(input_pixels) == width * height, "Keep width/height aligned with the PBS job."
    threads_per_block = stats["threads"]
    expected_sums = [sum(input_pixels[i:i + threads_per_block])
                     for i in range(0, len(input_pixels), threads_per_block)]
    reference_pixels, reference_mean, reference_gain = image_tools["adjust"](input_pixels, 128)
    assert block_sums == expected_sums
    assert stats["blocks"] == len(expected_sums)
    assert stats["sum"] == sum(input_pixels)
    assert math.isclose(stats["mean"], reference_mean, rel_tol=1e-12)
    assert math.isclose(stats["gain"], reference_gain, rel_tol=1e-12)
    assert gpu_pixels == reference_pixels, "GPU pixels differ from the CPU reference."
    for name, pixels in [("job-input.png", input_pixels), ("gpu-adjusted.png", gpu_pixels)]:
        image_tools["write_png"](preview_dir / name, pixels, width, height)
        display(Image(filename=str(preview_dir / name)))
    print("PASS: every GPU block sum and adjusted pixel matches the CPU reference.")

### Inspect the reduction on 16 sample pixels

This diagram is calculated on the CPU to illustrate the reduction; it does not
trace intermediate GPU values. All block threads still reach every barrier.

In [ ]:
sample_pixels = [preview[i * (len(preview) - 1) // 15] for i in range(16)]
display(SVG(data=image_tools["reduction_svg"](sample_pixels)))
print(f"Sample sum: {sum(sample_pixels)}")

Try another darkness, target brightness or thread count by editing the PBS job's
input-generation/run settings. Resubmit the documented job; rebuild in
`02-gpu.ipynb` only if CUDA source changed. Explain why unused threads in the
partial final block contribute zero but still reach every barrier.